# 第 8 章 確率変数と時系列: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-programming/08-random-variables.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
rho = 0.5         # AR(1) x(t) = ρ·x(t−1) + ε(t) の係数 ρ（0 は不可）
rho_other = 0.95  # 比べる相手の ρ（0 は不可）
sigma = 1.0       # ショック ε の標準偏差 σ
n = 100_000       # 系列の長さ（計算用）
n_plot = 100      # 図に描く系列の長さ
seed = 0          # 乱数の種（同じ種なら、同じ系列）
max_lag = 10      # 自己相関を、何ラグ先まで見るか

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and Path(getattr(_lib, "__file__", "") or "").resolve() != (_root / "lib" / "__init__.py").resolve():
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 理論値と、標本の値

In [ ]:
import numpy as np

from lib.chapter08.stochastic import (
    ar1_autocorrelation,
    ar1_series,
    ar1_variance,
    sample_autocorrelation,
    shock_half_life,
)

print(f"理論の分散（ρ = {rho}）: {ar1_variance(rho, sigma):.4f}")
for r in (rho, rho_other):
    x = ar1_series(r, sigma, n, np.random.default_rng(seed))
    print(
        f"ρ = {r}: 標本の分散 {x.var():.3f}、"
        f"ラグ 1 の自己相関 {sample_autocorrelation(x, 1):.3f}、"
        f"範囲 {x.min():.1f} から {x.max():.1f}、半減期 {shock_half_life(r):.1f} 期"
    )

## 図: 2 つの ρ の系列

In [ ]:
from lib.chapter08.plot import plot_series

plot_series(
    {
        f"rho = {r}": ar1_series(r, sigma, n_plot, np.random.default_rng(seed))
        for r in (rho, rho_other)
    }
)

## 図: 自己相関（標本と理論）

In [ ]:
from lib.chapter08.plot import plot_acf
from lib.chapter08.stochastic import autocorrelation_function

series = ar1_series(rho, sigma, n, np.random.default_rng(seed))
plot_acf(
    autocorrelation_function(series, max_lag),
    [ar1_autocorrelation(rho, lag) for lag in range(max_lag + 1)],
)

## 試してみよう

- `sigma = 2.0` にすると、分散の理論値は 1.3333 から 5.3333 に、
  4 倍になります（分散は σ の 2 乗に比例します）
- `rho = 0.9`、`rho_other = -0.5` にすると、ρ = 0.9 の分散は 5.2632、
  半減期は 6.6 期です。ρ = −0.5 のラグ 1 の自己相関は **−0.504（負）** で、系列は、
  上下に交互に動きます
- `seed = 1` にすると、ρ = 0.5 の標本の分散は 1.311 です（種が違うと、
  標本は少し変わりますが、理論の 1.3333 に近いままです）